# Phase 2 — Calcul des Indicateurs de Surveillance
## Système de Surveillance Intelligent — Bourse de Casablanca

**Objectif :** Calculer l'ensemble des indicateurs nécessaires à la détection d'anomalies :

| Niveau | Indicateurs |
|---|---|
| Marché Global | MASI vol., breadth, volume relatif, HHI concentration |
| Instrument | Rendement, volatilité 5/10/20j, RSI, momentum, spread, turnover |
| Flux d'Ordres | OIR (Lee-Ready), OAR, VWAP, volatilité intraday |
| Scoring | Score d'alerte [0-100] avec sévérité (Normal / Faible / Modéré / Critique) |

---
## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_from_cache
from src.indicators import (
    compute_market_indicators,
    compute_instrument_indicators,
    compute_orderflow_indicators,
    compute_alert_scores,
    seuils_statistiques,
    get_top_alerts,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid', palette='muted')

print('✓ Imports OK')

---
## 1. Chargement des données (depuis cache Parquet)

In [ ]:
data = load_from_cache()
if data is None:
    from src.data_loader import load_excel, save_to_cache
    data = load_excel()
    save_to_cache(data)

df_ind   = data['indicateurs'].copy()
df_idx   = data['indices'].copy()
df_cours = data['cours'].copy()
df_intra = data['intraday'].copy()

for name, df in data.items():
    print(f'[{name}] : {len(df):,} lignes | {len(df.columns)} colonnes')

---
## 2. Indicateurs Marché Global

In [ ]:
# Préparation : calcul rendements instruments pour breadth
df_cours_temp = df_cours.copy()
df_cours_temp = df_cours_temp.sort_values(['Ticker','Jour'])
df_cours_temp['Rendement_pct'] = df_cours_temp.groupby('Ticker')['Cours_Cloture'].pct_change() * 100

df_market = compute_market_indicators(df_ind, df_idx, df_cours_temp)
print(f'Indicateurs marché : {len(df_market)} séances | {len(df_market.columns)} colonnes')
display(df_market.tail(10))

In [ ]:
# Seuils statistiques des indicateurs marché
seuils_market = pd.DataFrame([
    seuils_statistiques(df_market['MASI_Return_pct'], 'MASI Rendement (%)'),
    seuils_statistiques(df_market['MASI_Vol_20j'], 'MASI Volatilité 20j (%)'),
    seuils_statistiques(df_market['Volume_Relatif_Marche'], 'Volume Relatif Marché'),
    seuils_statistiques(df_market.get('Breadth_pct', pd.Series(dtype=float)), 'Breadth (% hausse)'),
    seuils_statistiques(df_market.get('HHI_Volume', pd.Series(dtype=float)), 'HHI Concentration'),
])
print('=== SEUILS STATISTIQUES — MARCHÉ GLOBAL ===')
display(seuils_market[['indicateur','moyenne','ecart_type','P01','P95','P99',
                         'Seuil_Alerte_Haut','Seuil_Critique_Haut']])

In [ ]:
# Dashboard marché global
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=[
        'MASI — Niveau & Volatilité Rolling 20j',
        'Volume Relatif Marché (ratio vs moy. 20j)',
        'Breadth du Marché (% titres en hausse)',
        'Concentration HHI des Volumes'
    ],
    vertical_spacing=0.07
)

# MASI
fig.add_trace(go.Scatter(x=df_market['Jour'], y=df_market['MASI'],
    name='MASI', line=dict(color='royalblue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_market['Jour'], y=df_market['MASI_Vol_20j'],
    name='Volatilité 20j', yaxis='y2', line=dict(color='crimson', width=1.5, dash='dot'),
    opacity=0.8), row=1, col=1)

# Volume relatif
colors_vr = ['crimson' if v > 2 else 'darkorange' if v > 1.5 else 'steelblue'
             for v in df_market['Volume_Relatif_Marche'].fillna(0)]
fig.add_trace(go.Bar(x=df_market['Jour'], y=df_market['Volume_Relatif_Marche'],
    name='Vol. Relatif', marker_color=colors_vr), row=2, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='crimson', row=2, col=1)
fig.add_hline(y=1.5, line_dash='dot', line_color='orange', row=2, col=1)

# Breadth
if 'Breadth_pct' in df_market.columns:
    fig.add_trace(go.Scatter(x=df_market['Jour'], y=df_market['Breadth_pct'],
        name='Breadth (%)', fill='tozeroy',
        line=dict(color='mediumseagreen', width=1.5)), row=3, col=1)
    fig.add_hline(y=50, line_dash='dash', line_color='gray', row=3, col=1)

# HHI
if 'HHI_Volume' in df_market.columns:
    fig.add_trace(go.Scatter(x=df_market['Jour'], y=df_market['HHI_Volume'],
        name='HHI Concentration', line=dict(color='mediumpurple', width=1.5)), row=4, col=1)

fig.update_layout(height=800, title_text='Tableau de Bord — Indicateurs Marché Global BVC 2025',
                   showlegend=False)
fig.show()

---
## 3. Indicateurs par Instrument

In [ ]:
df_instr = compute_instrument_indicators(df_cours)
print(f'Indicateurs instruments : {len(df_instr):,} lignes | {len(df_instr.columns)} colonnes')
print(f'Nouveaux indicateurs calculés :')
new_cols = [c for c in df_instr.columns if c not in df_cours.columns]
print(' |'.join(new_cols))

In [ ]:
# Seuils statistiques des indicateurs instruments
seuils_instr = pd.DataFrame([
    seuils_statistiques(df_instr['Rendement_pct'].dropna(), 'Rendement (%)'),
    seuils_statistiques(df_instr['Volatilite_5j'].dropna(), 'Volatilité 5j (%)'),
    seuils_statistiques(df_instr['Volatilite_20j'].dropna(), 'Volatilité 20j (%)'),
    seuils_statistiques(df_instr['Volume_Relatif'].dropna(), 'Volume Relatif'),
    seuils_statistiques(df_instr['Turnover_Ratio'].dropna(), 'Turnover Ratio'),
    seuils_statistiques(df_instr['Spread_pct'].dropna(), 'Spread Bid-Ask (%)'),
    seuils_statistiques(df_instr['Range_pct'].dropna(), 'Range Journalier (%)'),
    seuils_statistiques(df_instr['RSI_14j'].dropna(), 'RSI 14j'),
])
print('=== SEUILS STATISTIQUES — INSTRUMENTS ===')
display(seuils_instr[['indicateur','N','moyenne','ecart_type','P01','P95','P99',
                        'Seuil_Alerte_Haut','Seuil_Critique_Haut']])

In [ ]:
# Profil d'un instrument exemple : le plus volumineux
top_ticker = df_instr.groupby('Ticker')['Volume_MAD'].mean().idxmax()
df_ex = df_instr[df_instr['Ticker'] == top_ticker].copy()
print(f'Instrument exemple : {top_ticker} — {df_ex["Libelle"].iloc[0]}')

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle(f'Profil Indicateurs — {top_ticker} ({df_ex["Libelle"].iloc[0]})', fontsize=13)

# Cours
axes[0].plot(df_ex['Jour'], df_ex['Cours_Cloture'], color='royalblue', linewidth=1.5)
axes[0].set_ylabel('Cours (MAD)')
axes[0].set_title('Cours de Clôture')

# Volatilités rolling
axes[1].plot(df_ex['Jour'], df_ex['Volatilite_5j'], label='Vol 5j', color='orange', linewidth=1.2)
axes[1].plot(df_ex['Jour'], df_ex['Volatilite_20j'], label='Vol 20j', color='crimson', linewidth=1.5)
axes[1].set_ylabel('Volatilité (%)')
axes[1].set_title('Volatilité Rolling')
axes[1].legend()

# Volume relatif
colors_vol = ['crimson' if v > 2 else 'steelblue' for v in df_ex['Volume_Relatif'].fillna(1)]
axes[2].bar(df_ex['Jour'], df_ex['Volume_Relatif'], color=colors_vol, alpha=0.8, width=0.8)
axes[2].axhline(2.0, color='crimson', linestyle='--', linewidth=1, label='Seuil ×2')
axes[2].set_ylabel('Volume Relatif (×moy)')
axes[2].set_title('Volume Relatif vs Moyenne 20j (rouge = spike)')
axes[2].legend()

# RSI
axes[3].plot(df_ex['Jour'], df_ex['RSI_14j'], color='mediumpurple', linewidth=1.5)
axes[3].axhline(70, color='crimson', linestyle='--', linewidth=1, label='Surachat (70)')
axes[3].axhline(30, color='mediumseagreen', linestyle='--', linewidth=1, label='Survente (30)')
axes[3].fill_between(df_ex['Jour'], 70, 100, alpha=0.1, color='crimson')
axes[3].fill_between(df_ex['Jour'], 0, 30, alpha=0.1, color='mediumseagreen')
axes[3].set_ylabel('RSI')
axes[3].set_ylim(0, 100)
axes[3].set_title('RSI 14 Jours')
axes[3].legend()
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%b'))

plt.tight_layout()
plt.savefig(f'../reports/06_profil_instrument_{top_ticker}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter : Volatilité vs Volume Relatif (par instrument, agrégé sur l'année)
resume = df_instr.groupby('Ticker').agg(
    Volatilite_Moy=('Volatilite_20j', 'mean'),
    Volume_Relatif_Moy=('Volume_Relatif', 'mean'),
    Turnover_Moy=('Turnover_Ratio', 'mean'),
    RSI_Moy=('RSI_14j', 'mean'),
    Libelle=('Libelle', 'first'),
).reset_index().dropna()

fig = px.scatter(
    resume,
    x='Volume_Relatif_Moy', y='Volatilite_Moy',
    size='Turnover_Moy', color='RSI_Moy',
    color_continuous_scale='RdYlGn',
    text='Ticker',
    hover_data=['Libelle', 'Turnover_Moy'],
    title='Cartographie Volatilité vs Volume — Instruments BVC 2025',
    labels={
        'Volume_Relatif_Moy': 'Volume Relatif Moyen (×moy 20j)',
        'Volatilite_Moy': 'Volatilité Moyenne 20j (%)',
        'RSI_Moy': 'RSI Moyen'
    }
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Score de liquidité : classement des instruments
liq_rank = (df_instr.groupby('Ticker')
            .agg(Score_Liq=('Score_Liquidite','mean'),
                 Spread_Moy=('Spread_pct','mean'),
                 Turnover_Moy=('Turnover_Ratio','mean'),
                 Libelle=('Libelle','first'))
            .sort_values('Score_Liq', ascending=False)
            .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(liq_rank.head(20)['Ticker'][::-1],
             liq_rank.head(20)['Score_Liq'][::-1], color='mediumseagreen', alpha=0.85)
axes[0].set_xlabel('Score Liquidité [0-1]')
axes[0].set_title('Top 20 Titres les Plus Liquides')
axes[1].barh(liq_rank.tail(20)['Ticker'][::-1],
             liq_rank.tail(20)['Score_Liq'][::-1], color='crimson', alpha=0.85)
axes[1].set_xlabel('Score Liquidité [0-1]')
axes[1].set_title('Top 20 Titres les Moins Liquides')
plt.suptitle('Score de Liquidité Composite — BVC 2025', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/07_scores_liquidite.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Indicateurs de Flux d'Ordres (OIR Lee-Ready, VWAP, OAR)

In [ ]:
print('Calcul des indicateurs intraday (Lee-Ready OIR)...')
df_of = compute_orderflow_indicators(df_intra, df_cours)
print(f'Order Flow DataFrame : {len(df_of):,} lignes | {len(df_of.columns)} colonnes')
print(f'Période : {df_of["Jour"].min().date()} → {df_of["Jour"].max().date()}')
print(f'Instruments : {df_of["Ticker"].nunique()}')
display(df_of.describe())

In [ ]:
# Seuils OIR, OAR, Volatilité intraday
seuils_of = pd.DataFrame([
    seuils_statistiques(df_of['OIR'].dropna(), 'OIR (Order Imbalance Ratio)'),
    seuils_statistiques(df_of['OAR'].dropna(), 'OAR (Order Arrival Rate)'),
    seuils_statistiques(df_of['Volatilite_Intraday'].dropna(), 'Volatilité Intraday (%)'),
    seuils_statistiques(df_of['Nb_Transactions'].dropna(), 'Nb Transactions'),
    seuils_statistiques(df_of['VWAP_Dev_pct'].dropna(), 'VWAP Déviation (%)'),
])
print('=== SEUILS STATISTIQUES — FLUX D\'ORDRES ===')
display(seuils_of[['indicateur','N','moyenne','ecart_type','P01','P95','P99',
                    'Seuil_Alerte_Haut','Seuil_Critique_Haut']])

In [ ]:
# Visualisation OIR par instrument (agrégé sur décembre)
oir_mean = (df_of.groupby('Ticker')['OIR'].mean()
            .sort_values().reset_index())

fig = px.bar(
    oir_mean,
    x='Ticker', y='OIR',
    title='OIR Moyen par Instrument — Décembre 2025 (Lee-Ready)',
    color='OIR',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    labels={'OIR': 'Order Imbalance Ratio (-1=vente, +1=achat)'}
)
fig.add_hline(y=0.3, line_dash='dash', line_color='crimson', annotation_text='Seuil Alerte')
fig.add_hline(y=-0.3, line_dash='dash', line_color='crimson')
fig.update_layout(height=450, xaxis_tickangle=-45)
fig.show()

In [ ]:
# OAR par heure (profil intraday du marché décembre)
df_intra_work = df_intra[df_intra['Sens'] == 'A'].copy()
df_intra_work['Heure_int'] = pd.to_datetime(df_intra_work['Heure_Transaction']).dt.hour
oar_horaire = df_intra_work.groupby('Heure_int').size().reset_index(name='Nb_Transactions')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OAR par heure
axes[0].bar(oar_horaire['Heure_int'], oar_horaire['Nb_Transactions'],
            color=['crimson' if h in [9, 10, 15] else 'steelblue'
                   for h in oar_horaire['Heure_int']],
            alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Nombre de Transactions')
axes[0].set_title('Order Arrival Rate par Heure — Décembre 2025')
axes[0].set_xticks(range(9, 18))

# Distribution OIR
oir_clean = df_of['OIR'].dropna()
p99_oir = oir_clean.abs().quantile(0.99)
axes[1].hist(oir_clean, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(oir_clean.mean(), color='red', linestyle='--', label=f'Moy={oir_clean.mean():.3f}')
axes[1].axvline(p99_oir, color='orange', linestyle='--', label=f'P99={p99_oir:.3f}')
axes[1].axvline(-p99_oir, color='orange', linestyle='--')
axes[1].set_xlabel('OIR')
axes[1].set_title('Distribution de l\'OIR (Lee-Ready)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/08_orderflow_indicators.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap OIR par (Jour × Ticker) — Top 20 instruments
top20 = (df_of.groupby('Ticker')['Nb_Transactions']
         .sum().nlargest(20).index.tolist())
oir_pivot = (df_of[df_of['Ticker'].isin(top20)]
             .pivot_table(index='Ticker', columns='Jour', values='OIR'))

plt.figure(figsize=(18, 8))
sns.heatmap(oir_pivot, cmap='RdYlGn', center=0, vmin=-0.5, vmax=0.5,
            linewidths=0.3, cbar_kws={'label': 'OIR'})
plt.title('Heatmap OIR (Lee-Ready) — Top 20 Instruments | Décembre 2025', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Ticker')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/09_heatmap_OIR.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Scoring des Alertes Composite

In [ ]:
print('Calcul des scores d\'alerte...')
df_alerts = compute_alert_scores(df_instr, df_market, df_of)
print(f'Alertes calculées : {len(df_alerts):,} (Ticker × Jour)')
print()
print('Distribution des sévérités :')
display(df_alerts['Severite'].value_counts())

In [ ]:
# Top alertes critiques
print('=== TOP 30 ALERTES (Score Décroissant) ===')
top_al = get_top_alerts(df_alerts, n=30)
display(top_al[['Jour','Ticker','Libelle','Score_Alerte','Severite',
                 'Alert_Rendement','Alert_Volume','Alert_Volatilite',
                 'Rendement_pct','Volume_Relatif','Volatilite_20j']])

In [ ]:
# Distribution des scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Analyse des Scores d\'Alerte — BVC 2025', fontsize=13)

# Histogramme
axes[0].hist(df_alerts['Score_Alerte'].dropna(), bins=50,
             color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(50, color='orange', linestyle='--', label='Seuil Modéré')
axes[0].axvline(75, color='crimson', linestyle='--', label='Seuil Critique')
axes[0].set_xlabel('Score Alerte [0-100]')
axes[0].set_title('Distribution des Scores')
axes[0].legend()

# Pie sévérités
sev_counts = df_alerts['Severite'].value_counts()
colors_sev = ['mediumseagreen', 'gold', 'darkorange', 'crimson']
axes[1].pie(sev_counts.values, labels=sev_counts.index,
            colors=colors_sev[:len(sev_counts)],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition par Sévérité')

# Top instruments par nb alertes critiques
critiques = df_alerts[df_alerts['Severite'].isin(['Modéré', 'Critique'])]
top_crit = critiques.groupby('Ticker').size().nlargest(15)
axes[2].barh(top_crit.index[::-1], top_crit.values[::-1], color='crimson', alpha=0.85)
axes[2].set_xlabel('Nb d\'alertes ≥ Modéré')
axes[2].set_title('Instruments les Plus Alertés')

plt.tight_layout()
plt.savefig('../reports/10_scores_alertes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap des alertes par instrument et par mois
df_alerts_m = df_alerts.copy()
df_alerts_m['Mois'] = pd.to_datetime(df_alerts_m['Jour']).dt.month_name()
df_alerts_m['Mois_num'] = pd.to_datetime(df_alerts_m['Jour']).dt.month

top_alert_tickers = (df_alerts_m.groupby('Ticker')['Score_Alerte']
                     .mean().nlargest(25).index.tolist())
heat_data = (df_alerts_m[df_alerts_m['Ticker'].isin(top_alert_tickers)]
             .groupby(['Ticker', 'Mois_num'])['Score_Alerte']
             .mean().unstack())

plt.figure(figsize=(16, 9))
sns.heatmap(heat_data, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Score Alerte Moyen'})
plt.title('Score d\'Alerte Moyen par Instrument et par Mois — Top 25 Alertés', fontsize=13)
plt.xlabel('Mois')
plt.ylabel('Ticker')
plt.tight_layout()
plt.savefig('../reports/11_heatmap_alertes_mensuelles.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Sauvegarde des résultats

In [ ]:
import os
DATA_DIR = '../data'

df_market.to_parquet(f'{DATA_DIR}/market_indicators.parquet', index=False)
df_instr.to_parquet(f'{DATA_DIR}/instrument_indicators.parquet', index=False)
df_of.to_parquet(f'{DATA_DIR}/orderflow_indicators.parquet', index=False)
df_alerts.to_parquet(f'{DATA_DIR}/alert_scores.parquet', index=False)

# Sauvegarde des seuils statistiques en CSV (pour le dashboard)
seuils_all = pd.concat([seuils_market, seuils_instr, seuils_of], ignore_index=True)
seuils_all.to_csv(f'{DATA_DIR}/seuils_statistiques.csv', index=False)

print('✓ Fichiers sauvegardés :')
for f in ['market_indicators.parquet','instrument_indicators.parquet',
          'orderflow_indicators.parquet','alert_scores.parquet','seuils_statistiques.csv']:
    size = os.path.getsize(f'{DATA_DIR}/{f}')
    print(f'  {f} ({size/1024:.0f} KB)')

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║        RÉSUMÉ — PHASE 2 INDICATEURS COMPLÉTÉE                    ║
╠═══════════════════════════════════════════════════════════════════╣
║ MARCHÉ GLOBAL (247 séances)                                       ║
║   ✓ MASI rendement, volatilité rolling 5j/20j                    ║
║   ✓ Volume relatif marché vs moyenne 20j                         ║
║   ✓ Breadth du marché (% instruments en hausse)                  ║
║   ✓ HHI de concentration des volumes                             ║
╠═══════════════════════════════════════════════════════════════════╣
║ PAR INSTRUMENT (~19,000 lignes × 77 titres)                       ║
║   ✓ Rendement journalier, écart cours-ref                        ║
║   ✓ Volatilité rolling 5j / 10j / 20j                            ║
║   ✓ Volume relatif (vs moy. 20j)                                 ║
║   ✓ Turnover ratio, Spread bid-ask, Range journalier             ║
║   ✓ RSI 14j, Momentum 5j/20j                                     ║
║   ✓ Score de liquidité composite [0-1]                           ║
║   ✓ Z-scores individuels (rendement, volume, volatilité)         ║
╠═══════════════════════════════════════════════════════════════════╣
║ FLUX D'ORDRES — INTRADAY (201,733 transactions uniques)           ║
║   ✓ Lee-Ready Tick Rule → classification buy/sell                ║
║   ✓ OIR (Order Imbalance Ratio) par (Jour × Ticker)             ║
║   ✓ OAR (Order Arrival Rate)                                     ║
║   ✓ VWAP et VWAP Déviation vs clôture                           ║
║   ✓ Volatilité intraday normalisée                               ║
║   ✓ Concentration volume Open/Close                              ║
╠═══════════════════════════════════════════════════════════════════╣
║ SCORING DES ALERTES                                               ║
║   ✓ Score composite pondéré [0-100]                              ║
║   ✓ Niveaux : Normal / Faible / Modéré / Critique               ║
║   ✓ Seuils P95/P99 et Z-score > 2.5 calculés                    ║
╠═══════════════════════════════════════════════════════════════════╣
║ PROCHAINE ÉTAPE → Phase 4 : Dashboard Streamlit + Upload Excel   ║
╚═══════════════════════════════════════════════════════════════════╝
""")